# Post-fault voltage behavior of converter-based ADNs

Six backend simulations compare short and long fault-clearing cases. Each panel uses its own event timing and labels converter-terminal voltage by control type: GFM or GFL. The CIGRE MV14 panels show grid, GFL bus 7, and GFM bus 9 voltages. Progress bars report both case-level and within-case simulation progress.

The full CIGRE EMT model has 94 states and is substantially slower than the reduced models. The notebook JIT-compiles the numerical derivatives, reuses each model for its paired case, and saves traces at 1 ms intervals; every fault schedule is still reapplied and checked before simulation.

In [12]:
from pathlib import Path
from time import perf_counter
import sys
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "codesimpy").is_dir())
sys.path.insert(0, str(ROOT))
from codesimpy.api import SystemFactory

cases = [
    dict(group="VSG-Infinite bus", stage="Short", system="VSGSMIBSystem", fault=dict(faultStart_s=0.20, faultComplete_s=0.201, recoveryStart_s=0.35, recoveryComplete_s=0.351, faultVoltage=0.10), p=dict(validationTime_s=1.0), buses=("VSG terminal", "Infinite bus"), labels=("GFM voltage", "Grid voltage")),
    dict(group="VSG-Infinite bus", stage="Long",  system="VSGSMIBSystem", fault=dict(faultStart_s=0.20, faultComplete_s=0.201, recoveryStart_s=0.70, recoveryComplete_s=0.701, faultVoltage=0.50), p=dict(validationTime_s=1.0), buses=("VSG terminal", "Infinite bus"), labels=("GFM voltage", "Grid voltage")),
    dict(group="GFL-Infinite bus", stage="Short", system="GFLLVRTSystem", fault=dict(faultStart_s=0.20, faultComplete_s=0.201, recoveryStart_s=0.35, recoveryComplete_s=0.351, faultVoltage=0.50), p=dict(validationTime_s=1.0), labels=("GFL voltage", "Grid voltage")),
    dict(group="GFL-Infinite bus", stage="Long",  system="GFLLVRTSystem", fault=dict(faultStart_s=0.20, faultComplete_s=0.201, recoveryStart_s=0.70, recoveryComplete_s=0.701, faultVoltage=0.50), p=dict(validationTime_s=1.0), labels=("GFL voltage", "Grid voltage")),
    dict(group="CIGRE MV14", stage="Short", system="CIGREMV14System", fault=dict(faultStart_s=0.20, faultComplete_s=0.201, recoveryStart_s=0.35, recoveryComplete_s=0.351, faultVoltage=0.10), p=dict(validationTime_s=1.0), buses=("source", "7", "9"), labels=("Grid voltage", "GFL voltage", "GFM voltage")),
    dict(group="CIGRE MV14", stage="Long",  system="CIGREMV14System", fault=dict(faultStart_s=0.20, faultComplete_s=0.201, recoveryStart_s=0.70, recoveryComplete_s=0.701, faultVoltage=0.50), p=dict(validationTime_s=1.0), buses=("source", "7", "9"), labels=("Grid voltage", "GFL voltage", "GFM voltage")),
]

In [ ]:
def selected_voltages(signals, case):
    if "busVoltageMagnitude" in signals:
        names, voltage = list(signals["busNames"]), np.asarray(signals["busVoltageMagnitude"])
        return tuple(voltage[:, names.index(bus)] for bus in case["buses"])
    return np.asarray(signals["pccVoltage"]), np.asarray(signals["gridVoltage"])


_model_cache = {}
RESULT_FILE = ROOT / "tests" / "adn_post_fault_voltage_results.npz"

def save_results(results):
    payload = {"signature": np.array(repr(cases))}
    for i, result in enumerate(results):
        for j, values in enumerate(result):
            payload[f"case_{i}_{j}"] = values
    np.savez_compressed(RESULT_FILE, **payload)

def load_results():
    with np.load(RESULT_FILE) as data:
        if data["signature"].item() != repr(cases):
            raise RuntimeError("Saved results do not match the current cases. Rerun the simulation cell.")
        return [tuple(data[f"case_{i}_{j}"] for j in range(1 + len(case["labels"])))
                for i, case in enumerate(cases)]

def run_case(case):
    requested = {**case["p"], **case["fault"]}
    cigre = case["system"] == "CIGREMV14System"
    solver = ({"method": "DOP853", "max_step": 2e-4, "rtol": 1e-3, "atol": 1e-6}
              if cigre else {})
    run_options = {"FaultMode": "time", "OutputStep_s": 1e-3,
                   "SolverSettings": solver}
    if case["system"] in _model_cache:
        system, applied = _model_cache[case["system"]]
        updated, options = SystemFactory.defaultParameters(case["system"], requested, run_options)
        applied.clear(); applied.update(updated)
        if "gridVoltage" in applied:
            system.setReference(str(system.devices["source"].uref[0]), applied["gridVoltage"])
        if system.eventName == "faultClearing":
            system.eventTime_s = applied["recoveryStart_s"]
    else:
        system, applied, options = SystemFactory.create(case["system"], requested, run_options)
        _model_cache[case["system"]] = system, applied
    assert all(np.isclose(applied[name], value) for name, value in case["fault"].items())
    point = system.initialize(options["StartTime_s"], ResidualTolerance=options["ResidualTolerance"])
    if not hasattr(system.rlGrid, "_numbaDerivative"):
        system.rlGrid.enableNumba()
    edges = np.linspace(options["StartTime_s"], options["EndTime_s"], 11)
    pieces, state = [], point["state"]
    with tqdm(total=options["EndTime_s"] - options["StartTime_s"],
              desc=f'{case["group"]} ({case["stage"].lower()})', unit='sim-s', leave=False) as bar:
        for start, stop in zip(edges[:-1], edges[1:]):
            all_breaks = np.asarray(options["Breakpoints_s"])
            breaks = all_breaks[(all_breaks > start) & (all_breaks < stop)]
            time = np.unique(np.r_[np.arange(start, stop, options["OutputStep_s"]), stop, breaks])
            result = system.simulate(time, InitialState=state, SolverSettings=solver,
                                     Breakpoints_s=breaks, saveSnapshot=False)
            traces = selected_voltages(result["signals"], case)
            keep = slice(None) if not pieces else slice(1, None)
            pieces.append((result["time_s"][keep], *(trace[keep] for trace in traces)))
            state = result["state"][-1]
            bar.update(stop - start)
    return tuple(np.concatenate(values) for values in zip(*pieces))

results, started = [], perf_counter()
case_bar = tqdm(cases, desc='Simulation cases', unit='case')
for case in case_bar:
    case_bar.set_postfix_str(f'{case["group"]}, {case["stage"].lower()} clearing')
    results.append(run_case(case))
save_results(results)
print(f'All simulations completed in {perf_counter() - started:.1f} s; results saved to {RESULT_FILE.name}')

In [ ]:
FONT = dict(family='Times New Roman', size=25, color='black')
titles = [f'({label}) {c["group"]}: t<sub>cl</sub> = {c["fault"]["recoveryStart_s"]:.2f} s'
          for label, c in zip('abcdef', cases)]
fig = make_subplots(rows=2, cols=3, 
                    vertical_spacing=0.125, horizontal_spacing=0.06)

if len(results) != len(cases):
    results = load_results()
if len(results) != len(cases):
    raise RuntimeError(f'Expected {len(cases)} results, received {len(results)}.')
colors = {"GFL voltage": "blue", "Grid voltage": "orange",
          "GFM voltage": "green"}
legend_rank = {"GFM voltage": 1, "GFL voltage": 2, "Grid voltage": 3}
for i, (case, result) in enumerate(zip(cases, results)):
    col, row = divmod(i, 2); row, col = row + 1, col + 1
    time, traces = result[0], result[1:]
    for j, (label, trace) in enumerate(zip(case['labels'], traces)):
        showlegend = i == 0 or (i == 4 and label == "GFL voltage")
        color = colors[label]
        fig.add_trace(go.Scatter(x=time, y=trace, name=label,
                                 line=dict(color=color, width=1.3),
                                 legendrank=legend_rank[label], showlegend=showlegend), row=row, col=col)
    fig.add_vrect(x0=case['fault']['faultStart_s'], x1=case['fault']['recoveryStart_s'], fillcolor='#BFBFBF', opacity=0.25, line_width=0, row=row, col=col)
    fig.update_xaxes(title_text='<i>t</i> (s)' if row == 3 else None, row=row, col=col)
    fig.update_yaxes(title_text='|<i>V</i>| (p.u.)' if col == 1 else None, rangemode='tozero', row=row, col=col)

axis = dict(showline=True, linecolor='black', linewidth=1, mirror=True, ticks='inside',
            ticklen=3, tickfont=FONT, title_font=FONT, title_standoff=3,
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5, zeroline=False)
fig.update_xaxes(**axis)
fig.update_yaxes(**axis)
fig.update_layout(template='plotly_white', width=1500, height=700, font=FONT,
                  margin=dict(l=80, r=20, t=65, b=70, pad=0), hovermode='x unified',
                  legend=dict(orientation='h', x=0.5, xanchor='center', y=1.025,
                              yanchor='bottom', font=FONT, bgcolor='rgba(255,255,255,0)'))
fig.for_each_annotation(lambda annotation: annotation.update(font=FONT))
fig.show(config={'displaylogo': False})

## Mechanism-focused cases: behaviors 1–2 and 6–7

These deliberately stressed cases identify mechanisms rather than represent a calibrated network. Behavior 2 displays both the unlimited current-reference magnitude |i<sub>unlim</sub>| and delivered filter-current magnitude |i|. LC/fast-mode oscillations are embedded in behaviors 1 and 2 by reducing the GFL capacitor damping conductance and the GFM virtual-resistance damping, so no separate behavior-3 figure is needed. The four figures retain the overview's Times New Roman styling, converter colors, boxed axes, and gray fault interval.

In [13]:
from copy import deepcopy

def _gfl_case(key, behavior, label, *, cache="gfl_common", scr=2.0, damping=0.8,
              k1=1.5, kp_pll=3.0, ki_pll=8.0, output_step=2e-4):
    return dict(
        key=key, behavior=behavior, label=label, cache=cache, system="GFLLVRTSystem",
        p=dict(SCR=scr, rho=0.1, Pref=1.0, Qref=0.0,
               filterCapacitance_pu=0.068, filterDampingConductance_pu=damping,
               validationTime_s=0.80, faultStart_s=0.20, faultComplete_s=0.201,
               recoveryStart_s=0.35, recoveryComplete_s=0.351, faultVoltage=0.35),
        controls=dict(gfl=dict(K1=k1, KpPll=kp_pll, KiPll=ki_pll),
                      capacitor=dict(Gd=damping, C=0.068)),
        output_step=output_step, max_step=2e-5,
    )

def _gfm_case(key, behavior, label, recovery, *, rho=0.1, rv=0.1,
              cache="gfm_common"):
    return dict(
        key=key, behavior=behavior, label=label, cache=cache,
        system="VSGSMIBSystem",
        p=dict(SCR=2.0, rho=rho, Pref=1.0, Vref=1.1,
               validationTime_s=1.0, faultStart_s=0.20, faultComplete_s=0.201,
               recoveryStart_s=recovery, recoveryComplete_s=recovery + 0.001,
               faultVoltage=0.30),
        controls=dict(vsg=dict(Imax=1.15, Kp=0.08, Lv=0.01, Rv=rv)),
        output_step=2e-4, max_step=2e-5,
    )

mechanism_cases = [
    _gfl_case("b1_gfl_recurrent", 1, "Recurrent GFL FRT", cache="gfl_recurrent",
              scr=1.5, damping=0.25, k1=4.0, kp_pll=0.2, ki_pll=0.2),
    _gfm_case("b2_gfm_recurrent", 2, "Recurrent GFM current limiting", 0.50,
              rho=0.10, rv=0.08, cache="gfm_low_damping"),
    _gfl_case("b6_pll_slow", 6, "Slow PLL", kp_pll=0.2, ki_pll=0.2),
    _gfl_case("b6_pll_fast", 6, "Fast PLL"),
    _gfm_case("b7_gfm_angle_limit", 7, "GFM angle/current-limit case", 0.65),
]

_mechanism_model_cache = {}

def _configure_controls(system, controls):
    for device_name, values in controls.items():
        for name, value in values.items():
            system.devices[device_name].p[name]["num"] = value

def _mechanism_signals(result, case, system):
    signals = result["signals"]
    data = dict(time=np.asarray(result["time_s"]),
                fault_start=case["p"]["faultStart_s"],
                recovery_start=case["p"]["recoveryStart_s"])
    if case["system"] == "GFLLVRTSystem":
        data.update(voltage=np.asarray(signals["pccVoltage"]),
                    grid_voltage=np.asarray(signals["gridVoltage"]),
                    lvrt=np.asarray(signals["lvrtMode"]),
                    current=np.asarray(signals["gflCurrentMagnitude"]),
                    pll_angle=np.asarray(signals["pllAngleError"]))
    else:
        indices = [i for i, name in enumerate(result["stateNames"])
                   if name.startswith("irefd") or name.startswith("irefq")]
        request = np.linalg.norm(result["state"][:, indices], axis=1)
        imax = float(system.devices["vsg"].p["Imax"]["num"])
        data.update(voltage=np.asarray(signals["busVoltageMagnitude"])[:, 0],
                    grid_voltage=np.asarray(signals["busVoltageMagnitude"])[:, 1],
                    current=np.asarray(signals["filterCurrentMagnitude"]),
                    iunlimited=request, current_limit=imax,
                    limiter_active=(request >= imax).astype(float),
                    delta_deg=np.rad2deg(np.unwrap(np.asarray(signals["delta"]))))
    return data

def run_mechanism_case(case):
    requested = deepcopy(case["p"])
    solver = dict(method="DOP853", max_step=case["max_step"], rtol=2e-5, atol=1e-8)
    run_options = dict(FaultMode="time", OutputStep_s=case["output_step"],
                       SolverSettings=solver)
    if case["cache"] in _mechanism_model_cache:
        system, applied = _mechanism_model_cache[case["cache"]]
        updated, options = SystemFactory.defaultParameters(case["system"], requested, run_options)
        applied.clear()
        applied.update(updated)
        system.parameters, system.parameterNames = applied, list(applied)
        if case["system"] == "GFLLVRTSystem":
            system.setReference(str(system.devices["source"].uref[0]), applied["gridVoltage"])
        if system.eventName == "faultClearing":
            system.eventTime_s = applied["recoveryStart_s"]
    else:
        system, applied, options = SystemFactory.create(case["system"], requested, run_options)
        _mechanism_model_cache[case["cache"]] = system, applied

    _configure_controls(system, case["controls"])
    point = system.initialize(options["StartTime_s"],
                              ResidualTolerance=options["ResidualTolerance"])
    if not hasattr(system.rlGrid, "_numbaDerivative"):
        system.rlGrid.enableNumba()
    time = np.unique(np.r_[np.arange(options["StartTime_s"], options["EndTime_s"],
                                     case["output_step"]),
                           options["EndTime_s"], options["Breakpoints_s"]])
    result = system.simulate(time, InitialState=point["state"],
                             SolverSettings=solver,
                             Breakpoints_s=options["Breakpoints_s"],
                             saveSnapshot=False)
    return _mechanism_signals(result, case, system)

mechanism_results = {}
started = perf_counter()
for case in tqdm(mechanism_cases, desc="Mechanism cases", unit="case"):
    mechanism_results[case["key"]] = run_mechanism_case(case)
print(f"Mechanism-focused simulations completed in {perf_counter() - started:.1f} s")

Mechanism cases: 100%|██████████| 5/5 [02:00<00:00, 24.07s/case]

Mechanism-focused simulations completed in 120.4 s


In [14]:
def _rising_entries(active, time, start):
    active, time = np.asarray(active, bool), np.asarray(time)
    return int(np.sum((~active[:-1]) & active[1:] & (time[1:] >= start)))

b1, b2 = mechanism_results["b1_gfl_recurrent"], mechanism_results["b2_gfm_recurrent"]
b6s, b6f = mechanism_results["b6_pll_slow"], mechanism_results["b6_pll_fast"]
b7 = mechanism_results["b7_gfm_angle_limit"]
post = lambda data: data["time"] >= data["recovery_start"]
print("Behavior 1 — post-fault LVRT entries:",
      _rising_entries(b1["lvrt"] > 0.5, b1["time"], b1["recovery_start"]))
print("Behavior 2 — post-fault limiter entries:",
      _rising_entries(b2["limiter_active"] > 0.5, b2["time"], b2["recovery_start"]))
print("Behavior 6 — voltage peaks (slow, fast PLL):",
      f'{np.max(b6s["voltage"][post(b6s)]):.3f}, {np.max(b6f["voltage"][post(b6f)]):.3f} p.u.')
print("Behavior 7 — post-fault GFM voltage range:",
      f'{np.min(b7["voltage"][post(b7)]):.3f}–{np.max(b7["voltage"][post(b7)]):.3f} p.u.')

Behavior 1 — post-fault LVRT entries: 18
Behavior 2 — post-fault limiter entries: 3
Behavior 6 — voltage peaks (slow, fast PLL): 1.457, 1.070 p.u.
Behavior 7 — post-fault GFM voltage range: 0.845–1.605 p.u.


In [15]:
MECH_FONT = dict(family="Times New Roman", size=25, color="black")
MECH_COLORS = dict(gfl="blue", grid="orange", gfm="green",
                   diagnostic="#A2142F", comparison="#7E2F8E")

def _style_mechanism_figure(fig, height, title):
    axis = dict(showline=True, linecolor="black", linewidth=1, mirror=True,
                ticks="inside", ticklen=3, tickfont=MECH_FONT,
                title_font=MECH_FONT, title_standoff=3, showgrid=True,
                gridcolor="#E6E6E6", gridwidth=0.5, zeroline=False)
    fig.update_xaxes(**axis)
    fig.update_yaxes(**axis)
    fig.update_layout(template="plotly_white", width=1200, height=height,
                      font=MECH_FONT,
                      title=dict(text=title, x=0.5, xanchor="center", font=MECH_FONT),
                      margin=dict(l=95, r=30, t=100, b=75, pad=0),
                      hovermode="x unified",
                      legend=dict(orientation="h", x=0.5, xanchor="center",
                                  y=1.025, yanchor="bottom", font=MECH_FONT,
                                  bgcolor="rgba(255,255,255,0)"))
    fig.for_each_annotation(lambda a: a.update(font=MECH_FONT))

def _shade_fault(fig, data, rows):
    for row in range(1, rows + 1):
        fig.add_vrect(x0=data["fault_start"], x1=data["recovery_start"],
                      fillcolor="#BFBFBF", opacity=0.25, line_width=0,
                      row=row, col=1)

def _active_intervals(time, active):
    time, active = np.asarray(time), np.asarray(active, bool)
    edges = np.diff(np.r_[False, active, False].astype(int))
    starts, stops = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
    return [(time[a], time[min(b, len(time) - 1)]) for a, b in zip(starts, stops)]

In [16]:
data = mechanism_results["b1_gfl_recurrent"]
fig_b1 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                       vertical_spacing=0.08, row_heights=[0.68, 0.32])
fig_b1.add_trace(go.Scatter(x=data["time"], y=data["voltage"],
                 name="GFL voltage", line=dict(color=MECH_COLORS["gfl"], width=1.3)), row=1, col=1)
fig_b1.add_trace(go.Scatter(x=data["time"], y=data["grid_voltage"],
                 name="Grid voltage", line=dict(color=MECH_COLORS["grid"], width=1.3)), row=1, col=1)
fig_b1.add_hline(y=0.9, line=dict(color="black", width=1.2, dash="dash"),
                 annotation_text="<i>V</i><sub>th</sub>", row=1, col=1)
fig_b1.add_trace(go.Scatter(x=data["time"], y=data["lvrt"], name="LVRT active",
                 line=dict(color=MECH_COLORS["diagnostic"], width=1.5, shape="hv")), row=2, col=1)
_shade_fault(fig_b1, data, 2)
fig_b1.update_yaxes(title_text="|<i>V</i>| (p.u.)", row=1, col=1)
fig_b1.update_yaxes(title_text="LVRT", range=[-0.08, 1.08], tickvals=[0, 1], row=2, col=1)
fig_b1.update_xaxes(title_text="<i>t</i> (s)", row=2, col=1)
_style_mechanism_figure(fig_b1, 700,
    "Behavior 1 — Recurrent GFL ride-through with LC-mode oscillations")
fig_b1.show(config={"displaylogo": False})

In [17]:
data = mechanism_results["b2_gfm_recurrent"]
fig_b2 = make_subplots(rows=4, cols=1, shared_xaxes=True,
                       vertical_spacing=0.045,
                       row_heights=[0.36, 0.24, 0.24, 0.16])
fig_b2.add_trace(go.Scatter(x=data["time"], y=data["voltage"],
                 name="GFM voltage",
                 line=dict(color=MECH_COLORS["gfm"], width=1.3)), row=1, col=1)
fig_b2.add_trace(go.Scatter(x=data["time"], y=data["grid_voltage"],
                 name="Grid voltage",
                 line=dict(color=MECH_COLORS["grid"], width=1.3)), row=1, col=1)
fig_b2.add_trace(go.Scatter(x=data["time"], y=data["iunlimited"],
                 name="|i<sub>unlim</sub>|",
                 line=dict(color=MECH_COLORS["diagnostic"], width=1.3)), row=2, col=1)
fig_b2.add_hline(y=data["current_limit"],
                 line=dict(color="black", width=1.2, dash="dash"),
                 annotation_text="<i>I</i><sub>max</sub>", row=2, col=1)
fig_b2.add_trace(go.Scatter(x=data["time"], y=data["current"],
                 name="|i|", line=dict(color=MECH_COLORS["gfm"], width=1.3)),
                 row=3, col=1)
fig_b2.add_hline(y=data["current_limit"],
                 line=dict(color="black", width=1.2, dash="dash"),
                 annotation_text="<i>I</i><sub>max</sub>", row=3, col=1)
fig_b2.add_trace(go.Scatter(x=data["time"], y=data["limiter_active"],
                 name="Limiter active",
                 line=dict(color=MECH_COLORS["diagnostic"], width=1.5, shape="hv"),
                 fill="tozeroy", fillcolor="rgba(162,20,47,0.12)"), row=4, col=1)
_shade_fault(fig_b2, data, 4)
fig_b2.update_yaxes(title_text="|<i>V</i>| (p.u.)", row=1, col=1)
fig_b2.update_yaxes(title_text="|<i>i</i><sub>unlim</sub>| (p.u.)",
                    rangemode="tozero", row=2, col=1)
fig_b2.update_yaxes(title_text="|<i>i</i>| (p.u.)",
                    rangemode="tozero", row=3, col=1)
fig_b2.update_yaxes(title_text="Limit", range=[-0.08, 1.08],
                    tickvals=[0, 1], row=4, col=1)
fig_b2.update_xaxes(title_text="<i>t</i> (s)", row=4, col=1)
_style_mechanism_figure(fig_b2, 980,
    "Behavior 2 — Recurrent GFM current limiting with LC-mode oscillations")
fig_b2.show(config={"displaylogo": False})

In [ ]:
slow, fast = mechanism_results["b6_pll_slow"], mechanism_results["b6_pll_fast"]
fig_b6 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                       vertical_spacing=0.08, row_heights=[0.58, 0.42])
fig_b6.add_trace(go.Scatter(x=slow["time"], y=slow["voltage"],
                 name="Slow PLL: voltage", line=dict(color=MECH_COLORS["diagnostic"], width=1.4)), row=1, col=1)
fig_b6.add_trace(go.Scatter(x=fast["time"], y=fast["voltage"],
                 name="Fast PLL: voltage", line=dict(color=MECH_COLORS["gfl"], width=1.4)), row=1, col=1)
fig_b6.add_trace(go.Scatter(x=slow["time"], y=slow["grid_voltage"],
                 name="Grid voltage", line=dict(color=MECH_COLORS["grid"], width=1.2)), row=1, col=1)
fig_b6.add_trace(go.Scatter(x=slow["time"], y=slow["pll_angle"],
                 name="Slow PLL: angle error", line=dict(color=MECH_COLORS["diagnostic"], width=1.4)), row=2, col=1)
fig_b6.add_trace(go.Scatter(x=fast["time"], y=fast["pll_angle"],
                 name="Fast PLL: angle error", line=dict(color=MECH_COLORS["gfl"], width=1.4)), row=2, col=1)
_shade_fault(fig_b6, slow, 2)
fig_b6.update_yaxes(title_text="|<i>V</i>| (p.u.)", rangemode="tozero", row=1, col=1)
fig_b6.update_yaxes(title_text="PLL angle error (deg)", row=2, col=1)
fig_b6.update_xaxes(title_text="<i>t</i> (s)", row=2, col=1)
_style_mechanism_figure(fig_b6, 750,
    "Behavior 6 — GFL post-fault overvoltage governed by PLL dynamics")
fig_b6.show(config={"displaylogo": False})

In [ ]:
data = mechanism_results["b7_gfm_angle_limit"]
fig_b7 = make_subplots(rows=3, cols=1, shared_xaxes=True,
                       vertical_spacing=0.055, row_heights=[0.43, 0.32, 0.25])
fig_b7.add_trace(go.Scatter(x=data["time"], y=data["voltage"],
                 name="GFM voltage", line=dict(color=MECH_COLORS["gfm"], width=1.4)), row=1, col=1)
fig_b7.add_trace(go.Scatter(x=data["time"], y=data["grid_voltage"],
                 name="Grid voltage", line=dict(color=MECH_COLORS["grid"], width=1.2)), row=1, col=1)
fig_b7.add_trace(go.Scatter(x=data["time"], y=data["delta_deg"],
                 name="GFM angle", line=dict(color=MECH_COLORS["comparison"], width=1.4)), row=2, col=1)
fig_b7.add_trace(go.Scatter(x=data["time"], y=data["current"],
                 name="Filter current", line=dict(color=MECH_COLORS["gfm"], width=1.3)), row=3, col=1)
fig_b7.add_hline(y=data["current_limit"], line=dict(color="black", width=1.2, dash="dash"),
                 annotation_text="<i>I</i><sub>max</sub>", row=3, col=1)
_shade_fault(fig_b7, data, 3)
for start, stop in _active_intervals(data["time"], data["limiter_active"] > 0.5):
    if stop >= data["recovery_start"]:
        for row in range(1, 4):
            fig_b7.add_vrect(x0=max(start, data["recovery_start"]), x1=stop,
                             fillcolor=MECH_COLORS["diagnostic"], opacity=0.07,
                             line_width=0, row=row, col=1)
fig_b7.update_yaxes(title_text="|<i>V</i>| (p.u.)", rangemode="tozero", row=1, col=1)
fig_b7.update_yaxes(title_text="<i>δ</i> (deg)", row=2, col=1)
fig_b7.update_yaxes(title_text="|<i>I</i>| (p.u.)", rangemode="tozero", row=3, col=1)
fig_b7.update_xaxes(title_text="<i>t</i> (s)", row=3, col=1)
_style_mechanism_figure(fig_b7, 850,
    "Behavior 7 — GFM under/overvoltage from angle motion and current limiting")
fig_b7.show(config={"displaylogo": False})